# The Same Agent, Using CrewAIThird time. In notebook 1 you wrote the agent loop by hand. In notebook 2 you rebuiltit on Microsoft Agent Framework. Now the same bug, the same three tools, and the sametask on **CrewAI**.Doing this twice is the point. One framework looks like *the* way to build agents. Twoframeworks disagreeing about what an agent even *is* makes it obvious that the loop youwrote in notebook 1 is the durable part and everything above it is a matter of taste.CrewAI takes a noticeably stronger position than Agent Framework did. It thinks interms of **crews**: agents with personas, assigned to tasks, working a process. We aregoing to use exactly one agent for one task, which cuts against the grain — and thatis informative in itself.**What you need:** the same `.env` you set up for notebook 1, with your`OPENROUTER_API_KEY` in it. Nothing new to configure.

In [ ]:
%pip install --quiet crewai python-dotenv pytestimport osimport subprocessimport sysfrom pathlib import Pathfrom dotenv import find_dotenv, load_dotenvfrom crewai import LLM, Agent, Crew, Process, Taskfrom crewai.tools import toolfrom pydantic import Field# CrewAI routes by provider prefix: "openrouter/" tells it where to send the request,# and the rest is the model slug OpenRouter expects. ":free" is the no-cost,# rate-limited endpoint - drop it for the paid one if the rate limit gets in your way.MODEL = "openrouter/nvidia/nemotron-3.5-lightning:free"WORKDIR = Path.cwd()load_dotenv(find_dotenv())llm = LLM(    model=MODEL,    api_key=os.environ["OPENROUTER_API_KEY"],    base_url="https://openrouter.ai/api/v1",)print("provider:", llm.provider)print("model   :", llm.model)print("workdir :", WORKDIR)

## 1. Does the API work?`llm.call()` is CrewAI's plain request method — no agent, no task, no crew. One request,one string back.Note there is no `await` here. CrewAI is synchronous by default, unlike Agent Framework.

In [ ]:
print(llm.call("Write a haiku about debugging code."))

## 2. The taskIdentical to both previous notebooks.> **Re-run the `%%writefile buggy.py` cell to put the bug back** and start clean.

In [ ]:
%%writefile buggy.pydef add_reading(reading, log=[]):    """Append a sensor reading to a log and return the log."""    log.append(reading)    return logdef average(readings):    """Return the mean of a list of readings."""    return sum(readings) / len(readings)

In [ ]:
%%writefile test_buggy.pyfrom buggy import add_reading, averagedef test_average():    assert average([2, 4, 6]) == 4def test_logs_are_independent():    first = add_reading(1)    second = add_reading(2)    assert first == [1]    assert second == [2]

In [ ]:
print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 3. The toolsSame three functions again. CrewAI has its own `@tool` decorator, and it builds theschema from the signature much as Agent Framework did.**One difference that will silently cost you accuracy.** Agent Framework read parameterdescriptions from `Annotated[str, "..."]`. CrewAI ignores `Annotated` entirely — itreads descriptions from a pydantic `Field` used as the default value. Get this wrong andeverything still runs; the model just never learns that `old` has to match exactly once,and it makes more failed edits.This is a good thing to be annoyed by. The concept is identical in both frameworks andthe spelling is arbitrary and incompatible, which is precisely why the loop you wrote innotebook 1 was worth writing.

In [ ]:
@tool("read_file")def read_file(    path: str = Field(..., description="File to read, e.g. buggy.py"),) -> str:    """Read the full contents of a file in the working directory."""    return (WORKDIR / path).read_text()@tool("edit_file")def edit_file(    path: str = Field(..., description="File to edit, e.g. buggy.py"),    old: str = Field(..., description="Exact text to replace. Must appear exactly once, whitespace included."),    new: str = Field(..., description="Replacement text."),) -> str:    """Replace an exact snippet of text in a file."""    p = WORKDIR / path    text = p.read_text()    if text.count(old) == 0:        return "ERROR: 'old' not found in the file. Read it again and match it exactly."    if text.count(old) > 1:        return "ERROR: 'old' appears more than once. Include more surrounding context."    p.write_text(text.replace(old, new))    return f"ok, edited {path}"@tool("run_tests")def run_tests() -> str:    """Run the pytest suite and return its output."""    result = subprocess.run(        [sys.executable, "-m", "pytest", "-q"],        capture_output=True, text=True, timeout=60, cwd=WORKDIR,    )    return (result.stdout + result.stderr)[-2000:] or "(no output)"# Check that the descriptions actually made it into the schema:print(edit_file.args_schema.model_json_schema()["properties"])

## 4. Watching the tool callsIn notebook 1 you printed the tool calls yourself. In notebook 2 you had to writemiddleware to get them back. CrewAI takes a third position: with `verbose=True` it printsa very loud, heavily formatted transcript whether you asked for one or not.That is convenient and it is also somebody else's idea of what matters. `step_callback`is the structured hook underneath it — it fires after every agent step with either an`AgentAction` (the model asked for a tool) or an `AgentFinish` (it is done).Three frameworks, three answers to "how do I see what my agent did". None of them changedwhat the agent actually does.

In [ ]:
from crewai.agents.parser import AgentAction, AgentFinishdef log_step(step):    # AgentAction  has: thought, tool, tool_input, text, result    # AgentFinish  has: thought, output, text    # TODO 1: if step is an AgentAction, print the tool name and tool_input,    #         then print its result truncated to a few hundred characters.    # TODO 2: if step is an AgentFinish, print the final output.    pass

## 5. Agent, Task, CrewHere is where CrewAI's opinion shows. Agent Framework wanted one object. CrewAI wantsthree, and it wants you to describe your agent as a *person*.**Agent** — `role`, `goal` and `backstory`. Be clear-eyed about what this is: threefields that get concatenated into a system prompt. It is the same `SYSTEM` string fromnotebook 1, in a costume. Whether the costume helps is a real question and not a settledone; it does reliably make people write better prompts than they otherwise would.**Task** — what to do, plus `expected_output`. This second field is CrewAI's genuinelygood idea. Being made to state what a finished answer looks like is a rubric, and ittends to improve results in a way the persona fields do not.**Crew** — the agents, the tasks and a process. With one agent and one task there isnothing to orchestrate, so this is pure ceremony here. It stops being ceremony themoment you add a second agent, which is what CrewAI is actually for.Note `max_iter=8`. **CrewAI's default is 25** — three times the cap you chose deliberatelyin notebook 1. Setting it explicitly is not optional if you care what you spend.

In [ ]:
# TODO 3: build the agent.#   Agent(role=..., goal=..., backstory=..., tools=[read_file, edit_file, run_tests],#         llm=llm, verbose=True, max_iter=8, step_callback=log_step)debugger = ...# TODO 4: build the task. Give it a description and an expected_output.#   Task(description=..., expected_output=..., agent=debugger)fix_the_bug = ...# TODO 5: build the crew.#   Crew(agents=[debugger], tasks=[fix_the_bug], process=Process.sequential, verbose=True)crew = ...

## 6. Run it`kickoff()` is CrewAI's `run()`. Expect a great deal of output — that is `verbose=True`doing its thing, with your `step_callback` lines threaded through it.

In [ ]:
result = crew.kickoff()print("\n=== result ===")print(result)

## 7. Did it actually change the file?

In [ ]:
print(Path("buggy.py").read_text())print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 8. Three frameworks, one loop| | By hand | Agent Framework | CrewAI ||---|---|---|---|| Unit of work | your `for` loop | `Agent` | `Agent` + `Task` + `Crew` || Tool schemas | ~40 lines you wrote | `@tool` + `Annotated` | `@tool` + `Field` || Param descriptions | in your JSON | `Annotated[str, "..."]` | `Field(description=...)` || Running it | your loop | `await agent.run()` | `crew.kickoff()` || Async? | your choice | async | sync || Seeing tool calls | your `print` | function middleware | `verbose=True`, `step_callback` || Turn cap | `max_turns=8`, yours | an unread default | `max_iter`, default **25** || Prompt | one `SYSTEM` string | `instructions` | `role` + `goal` + `backstory` |**What is identical in all three columns:** the model emits a request, your function runsit, the result goes back, repeat. Everything else in this table is packaging.That is the whole workshop. Frameworks are worth using — they give you retries, tracing,multi-agent orchestration and MCP for free. But they disagree with each other aboutalmost everything except the loop, they change fast (Agent Framework exists becauseMicrosoft deprecated the two frameworks everyone taught eighteen months ago), and everyone of them has defaults deciding things you would otherwise decide yourself.You now know what is underneath. That knowledge outlives all three columns.### Try this1. Remove `run_tests` from the agent's `tools` list. Third time you have run this   experiment, in a third framework.2. Set `max_iter=2` and watch what happens when it runs out.3. Delete `step_callback=log_step` and keep `verbose=True`. Whose transcript do you   prefer, and what does CrewAI's leave out?